# AI Avatar Studio — Google Colab GPU Worker

這個 Notebook 會在 Google Colab 的免費/付費 GPU 上啟動一個遠端推論伺服器，提供：
1. **聲音克隆與文字轉語音 (`POST /api/tts`)**：使用 **F5-TTS**（支援中英文零樣本聲音克隆）
2. **人物對嘴影片生成 (`POST /api/lipsync`)**：使用 **Wav2Lip**（支援單張人像照片或影片 + 語音對嘴）

### 使用步驟
1. 在 Colab 上方選單點選 **「執行階段」→「變更執行階段類型」→ 選擇 `T4 GPU`**（或更高等級 GPU）。
2. 依序執行下方 3 個儲存格。
3. 第 3 個儲存格會印出一個 `https://xxxx.trycloudflare.com` 網址，複製它貼回本機網頁右上角的 **「Colab 設定」** 即可開始生成！

In [ ]:
#@title 1. 檢查 GPU 與安裝基礎套件 (約 2~3 分鐘)
!nvidia-smi
!pip install -q fastapi uvicorn python-multipart f5-tts soundfile pydub
!apt-get -qq update && apt-get -qq install -y ffmpeg

# 下載 cloudflared (免註冊即可建立公開 HTTPS Tunnel)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb
print("✅ 基礎套件安裝完成")

In [ ]:
#@title 2. 下載 Wav2Lip 對嘴模型與權重
import os
if not os.path.exists("/content/Wav2Lip"):
    !git clone -q https://github.com/Rudrabha/Wav2Lip.git /content/Wav2Lip
    !pip install -q librosa==0.9.2 opencv-python-headless scipy tqdm numba

os.makedirs("/content/Wav2Lip/checkpoints", exist_ok=True)
os.makedirs("/content/Wav2Lip/face_detection/detection/sfd", exist_ok=True)

if not os.path.exists("/content/Wav2Lip/checkpoints/wav2lip_gan.pth"):
    !wget -q -O /content/Wav2Lip/checkpoints/wav2lip_gan.pth "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth"
if not os.path.exists("/content/Wav2Lip/face_detection/detection/sfd/s3fd.pth"):
    !wget -q -O /content/Wav2Lip/face_detection/detection/sfd/s3fd.pth "https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth"

print("✅ Wav2Lip 模型準備完成")

In [ ]:
#@title 3. 啟動 FastAPI GPU Worker 並產生公開連線網址
import os
import re
import shutil
import subprocess
import threading
import time
import uuid
from pathlib import Path
import torch
import uvicorn
from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse

# 選填：若需要保護 API，可設定自訂 Token，並於本機平台填入相同 Token
API_TOKEN = ""  #@param {type:"string"}

WORK_DIR = Path("/content/worker_tmp")
WORK_DIR.mkdir(parents=True, exist_ok=True)

app = FastAPI(title="Colab AI Avatar GPU Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

def verify_token(authorization: str | None):
    if API_TOKEN.strip():
        expected = f"Bearer {API_TOKEN.strip()}"
        if authorization != expected:
            raise HTTPException(status_code=401, detail="Invalid API token")

@app.get("/health")
def health(authorization: str | None = Header(default=None)):
    verify_token(authorization)
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
    return {"status": "ok", "gpu": gpu_name, "engines": ["F5-TTS", "Wav2Lip-GAN"]}

@app.post("/api/tts")
async def tts_clone(
    text: str = Form(...),
    language: str = Form("zh"),
    speed: float = Form(1.0),
    ref_text: str = Form(""),
    ref_audio: UploadFile = File(...),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    raw_ref = WORK_DIR / f"{req_id}_raw_{ref_audio.filename}"
    wav_ref = WORK_DIR / f"{req_id}_ref.wav"
    out_dir = WORK_DIR / f"{req_id}_tts_out"
    out_dir.mkdir(exist_ok=True)

    raw_ref.write_bytes(await ref_audio.read())
    subprocess.run(["ffmpeg", "-y", "-i", str(raw_ref), "-ar", "24000", "-ac", "1", str(wav_ref)], check=True)

    cmd = [
        "f5-tts_infer-cli",
        "--model", "F5TTS_v1_Base",
        "--ref_audio", str(wav_ref),
        "--gen_text", text,
        "--speed", str(speed),
        "--output_dir", str(out_dir),
        "--output_file", "output.wav",
    ]
    if ref_text.strip():
        cmd.extend(["--ref_text", ref_text.strip()])

    proc = subprocess.run(cmd, capture_output=True, text=True)
    out_wav = out_dir / "output.wav"
    if proc.returncode != 0 or not out_wav.exists():
        raise HTTPException(status_code=500, detail=f"TTS failed: {proc.stderr[-500:]}")

    return FileResponse(str(out_wav), media_type="audio/wav", filename="output.wav")

@app.post("/api/lipsync")
async def lipsync(
    avatar: UploadFile = File(...),
    audio: UploadFile = File(...),
    authorization: str | None = Header(default=None),
):
    verify_token(authorization)
    req_id = uuid.uuid4().hex[:8]
    ext = Path(avatar.filename or "face.png").suffix.lower() or ".png"
    face_path = WORK_DIR / f"{req_id}_face{ext}"
    raw_audio = WORK_DIR / f"{req_id}_audio_raw.wav"
    wav_audio = WORK_DIR / f"{req_id}_audio_16k.wav"
    out_mp4 = WORK_DIR / f"{req_id}_result.mp4"

    face_path.write_bytes(await avatar.read())
    raw_audio.write_bytes(await audio.read())
    subprocess.run(["ffmpeg", "-y", "-i", str(raw_audio), "-ar", "16000", "-ac", "1", str(wav_audio)], check=True)

    cmd = [
        "python", "/content/Wav2Lip/inference.py",
        "--checkpoint_path", "/content/Wav2Lip/checkpoints/wav2lip_gan.pth",
        "--face", str(face_path),
        "--audio", str(wav_audio),
        "--outfile", str(out_mp4),
        "--nosmooth",
    ]
    proc = subprocess.run(cmd, cwd="/content/Wav2Lip", capture_output=True, text=True)
    if proc.returncode != 0 or not out_mp4.exists():
        raise HTTPException(status_code=500, detail=f"LipSync failed: {proc.stderr[-500:]}")

    return FileResponse(str(out_mp4), media_type="video/mp4", filename="result.mp4")

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8899, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

print("🚀 正在啟動 Cloudflare Tunnel，請稍候...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8899"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in tunnel.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("\n" + "=" * 60)
        print("✅ Colab GPU Worker 已上線！請複製以下網址貼到本機設定：")
        print(match.group(0))
        print("=" * 60 + "\n")